In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *


####Datat Reading

In [0]:
df=spark.read.format("parquet").load("abfss://bronze@datalakedatabricksram.dfs.core.windows.net/customers")
df.display()


In [0]:
df=df.drop("_rescued_data")
df.display()

In [0]:
df=df.withColumn("domains",split(col("email"),"@")[1])
df.display()

In [0]:
df.groupBy("domains").agg(count("customer_id").alias("total_customers")).sort("total_customers",ascending=False).display()


In [0]:
df=df.withColumn("full_name",concat(col("first_name"),lit(" "),col("last_name")))

df=df.drop("first_name","last_name")
df.display()

####Data Writing

In [0]:
df.write.format('delta').\
    mode('overwrite').\
    save('abfss://silver@datalakedatabricksram.dfs.core.windows.net/customers')


In [0]:
%sql 
create table if not exists databricks_catalog.silver.customers_silver
using delta
location 'abfss://silver@datalakedatabricksram.dfs.core.windows.net/customers'

In [0]:
%sql
select * from databricks_catalog.silver.customers_silver